# Final test: our model vs base Qwen3.5-2B on unseen questions

**Our model** = Qwen3.5-2B (non-thinking) + the `v1.2` LoRA adapter at half strength (`ckpt-00647-x50`). On the 1,008 development questions it beat base Qwen: 17.9% vs 14.5% (significant, p = 0.0035), with maths unchanged (69.4% vs 67.1%).

**Why one more test.** We picked this adapter by looking at the development questions (UGPhysics items ranked 0 to 99 in each subject), so its score there is slightly optimistic. The questions ranked **100 to 199** in each subject were never used for any decision. Our score on them is the honest number for the model card.
- GeometricalOptics has only 58 questions, all already used, so the test covers the other 12 subjects: 1,200 questions.
- Same harness, prompt and limits as before: greedy decoding, 4,096 new tokens.

**Who does what.** Every job can run at the same time on a different account. A job is safe to interrupt: rerun the job cell, on any account but only one at a time, and it continues.

| Account | `JOB` | What | GPU time |
|---|---|---|---|
| A | `base-test` | base Qwen on the 1,200 test questions | about 7 h (2 sessions) |
| B | `ours-test` | our model on the 1,200 test questions | about 5 h (1 to 2 sessions) |
| C | `math`, then `think` | MATH-500 from 70 to 210 questions for both models; then base Qwen in **thinking** mode on the 260 dev questions | about 1.5 h + 2 h |

`think` checks the other mode of the same model, so the model card can say which one we beat.

**Before you start:** put the new `physeval.py` (**version 1.4**) in the Drive folder. Runtime > Change runtime type > **T4 GPU**.

## 1. Settings

In [ ]:
WORKDIR = "/content/drive/MyDrive/physics_eval"
JOB = "base-test"        # "base-test" (account A), "ours-test" (account B), "math" or "think" (account C)

OURS = "ft-qwen-v1.2-lora-s647-x50"                              # our model's evaluation runs start with this name
ADAPTER = f"{WORKDIR}/train_runs/qwen-v1.2-lora/ckpt-00647-x50"  # the half-strength adapter (made in PhysicsTrain 9d)
OURS_FLAGS = f'--model Qwen/Qwen3.5-2B --enable-thinking false --adapter "{ADAPTER}"'
TEST = "ugphysics=100:200"   # held-out slice: ranks 100 to 199 in every subject
JOBS = {  # (run name, model flags, benchmark, which questions)
    "base-test": [("qwen35-2b-nothink-test", "", "ugphysics", TEST)],
    "ours-test": [(OURS + "-test", OURS_FLAGS, "ugphysics", TEST)],
    "math":      [("qwen35-2b-nothink", "", "math500", "math500=30"), (OURS, OURS_FLAGS, "math500", "math500=30")],
    "think":     [("qwen35-2b-think", "", "ugphysics", "ugphysics=20")],
}
assert JOB in JOBS, f"JOB must be one of {list(JOBS)}"

## 2. Mount Drive and check the files

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, sys, shutil
sys.path.insert(0, WORKDIR)
import physeval
assert physeval.VERSION >= "1.4", "replace physeval.py with version 1.4"
assert os.path.exists(f"{ADAPTER}/adapter_config.json"), "half-strength adapter missing (PhysicsTrain section 9d makes it)"
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 3. Install packages

In [ ]:
!pip -q uninstall -y torchao
!pip -q install -U transformers peft accelerate safetensors
!pip -q install "antlr4-python3-runtime==4.11.1" chardet

## 4. Run the job
Prints progress and an ETA. If the session ends, run sections 1 to 3 again (same `JOB`), then this cell; finished questions are kept.

In [ ]:
for run, flags, bench, per in JOBS[JOB]:
    print("=" * 30, run, bench, per)
    !python "{WORKDIR}/physeval.py" generate --workdir "{WORKDIR}" --run {run} {flags} \
        --benchmarks {bench} --per-group "{per}" --max-new-tokens 4096 --batch-size 16
    !python "{WORKDIR}/physeval.py" grade --workdir "{WORKDIR}" --run {run}
print("JOB", JOB, "finished")

## 5. Results (no GPU needed; any account, after the jobs finish)
Two reports, each saved under its own name in `results/`:
- `summary_final_test.md`: **the main result**, our model vs base Qwen on the 1,200 unseen questions
- `summary_dev_all.md`: development questions, with MATH-500 on 210 questions and base Qwen in thinking mode

Send me both.

In [ ]:
from IPython.display import Markdown, display
!python "{WORKDIR}/physeval.py" report --workdir "{WORKDIR}" --run qwen35-2b-nothink-test,{OURS}-test > /dev/null
shutil.copy(f"{WORKDIR}/results/summary.md", f"{WORKDIR}/results/summary_final_test.md")
!python "{WORKDIR}/physeval.py" report --workdir "{WORKDIR}" --run lfm25-1.2b-thinking,qwen35-2b-nothink,qwen35-2b-think,{OURS} > /dev/null
shutil.copy(f"{WORKDIR}/results/summary.md", f"{WORKDIR}/results/summary_dev_all.md")
display(Markdown(open(f"{WORKDIR}/results/summary_final_test.md").read()))
print("saved: results/summary_final_test.md and results/summary_dev_all.md")